In [ ]:
pip install seaborn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
%matplotlib inline


In [ ]:
df = pd.read_csv('Bengaluru_House_Data.csv')

In [ ]:
df.head()
df.shape
df.info()

In [ ]:
df.head()

In [ ]:
df.shape 

In [ ]:
df.isna().sum()

In [ ]:
for i in df.columns:
    print(df[i].value_counts())
    print("*"*20)

In [ ]:
df.drop(columns=['area_type', 'society', 'balcony', 'availability'], inplace=True)

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df['location'].value_counts()


In [ ]:
df['location'] = df['location'].fillna('Whitefield')

In [ ]:
df['size'].value_counts()

In [ ]:
df['size'] = df['size'].fillna('2 BHK')

In [ ]:
df['bath'] = df['bath'].fillna(df['bath'].median())

In [ ]:
df.info()

In [ ]:
df['bedroom'] = df['size'].str.split(' ').str.get(0).astype(int)

In [ ]:
df.head()

In [ ]:
df.drop(columns=['size'], inplace=True)

In [ ]:
df[df.bedroom >20]

In [ ]:
df.drop(df[df.bedroom >20].index, inplace=True)

In [ ]:
df.info

In [ ]:
df.head()

In [ ]:
df

In [ ]:
df['total_sqft'].unique()

In [ ]:
def convertrange(x):
    temp = x.split('-')
    if len(temp) == 2:
        return (float(temp[0]) + float(temp[1])) / 2
    try:
        return float(x)
    except:
        return None

In [ ]:
df['total_sqft'] = df['total_sqft'].apply(convertrange)

In [ ]:
df['total_sqft'].unique()

In [ ]:
df.info()

In [ ]:
df['total_sqft'] = df['total_sqft'].fillna(df['total_sqft'].median())


In [ ]:
df['ppsqft'] = df['price'] * 100000 / df['total_sqft']
df.head()

In [ ]:
df.describe()

In [ ]:
df['location'].value_counts()

In [ ]:
df['location'] = df['location'].str.strip()

In [ ]:
df['location'].value_counts()

In [ ]:

location_less_than_10 = df['location'].value_counts()[df['location'].value_counts() <= 10]
location_less_than_10

In [ ]:
df['location'] = df['location'].replace(location_less_than_10.index, 'other')
df['location'].value_counts()

In [ ]:
df = df[((df['total_sqft'] / df['bedroom']) >= 300)]
df.describe()

In [ ]:
df.shape

In [ ]:
def remove_outliers_sqrt(df):
    df_output = pd.DataFrame()
    for key,subdf in df.groupby('location'):
        mean = subdf['ppsqft'].mean()
        std = subdf['ppsqft'].std()
        reduced_df = subdf[(subdf['ppsqft'] > (mean - std)) & (subdf['ppsqft'] < (mean + std))]
        df_output = pd.concat([df_output, reduced_df], ignore_index=True)
    return df_output
df = remove_outliers_sqrt(df)
df.describe()

In [ ]:
def remove_bhkoutliers(df):
    exclude_indices = np.array([])
    for location, location_df in df.groupby('location'):
        bhk_stats = {}
        for bhk, bhk_df in location_df.groupby('bedroom'):
            bhk_stats[bhk] = {
                'mean': bhk_df['ppsqft'].mean(),
                'std': bhk_df['ppsqft'].std(),
                'count': bhk_df.shape[0]
            }
        for bhk, bhk_df in location_df.groupby('bedroom'):
            stats = bhk_stats.get(bhk - 1)
            if stats and stats['count'] > 5:
                exclude_indices = np.append(
                    exclude_indices,
                    bhk_df[bhk_df['ppsqft'] < (stats['mean'])].index.values
                )
    return df.drop(exclude_indices, axis='index')  # Correctly indented

In [ ]:
df = remove_bhkoutliers(df)

In [ ]:
df.shape

In [ ]:
df = df.drop(columns=['ppsqft'])

In [ ]:
df.head()

In [ ]:
df.to_csv('cleaned_bengaluru_house_data.csv')

ml



In [ ]:
x = df.drop(columns=['price'])
y = df['price']
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score



In [ ]:
X_tarin,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)


In [ ]:
print(X_tarin.shape, x_test.shape, y_train.shape, y_test.shape)

In [ ]:


columntrans = make_column_transformer((OneHotEncoder(sparse_output=False), ['location']),
    remainder='passthrough'
)




In [ ]:
scaler = StandardScaler()

In [ ]:
lr = LinearRegression()

In [ ]:
model = make_pipeline(columntrans, scaler, lr)

In [ ]:
model.fit(X_tarin, y_train)

In [ ]:
y_pred = model.predict(x_test)

In [ ]:
r2 = r2_score(y_test, y_pred)
r2

In [ ]:
lasso = Lasso()

In [ ]:
model = make_pipeline(columntrans, scaler,lasso)
model.fit(X_tarin, y_train)
y_pred_lasso = model.predict(x_test)

In [ ]:
r2_score(y_test, y_pred_lasso)

In [ ]:
ridge = Ridge()
model = make_pipeline(columntrans, scaler, ridge)
model.fit(X_tarin, y_train)
y_pred_ridge = model.predict(x_test)
r2_score(y_test, y_pred_ridge)


In [ ]:
import pickle
pickle.dump(model,open('ridgemodel.pkl', 'wb'))

In [ ]:
with open('column_transformer.pkl', 'wb') as f:
    pickle.dump(columntrans, f)